#Run StructureMap software

Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import tempfile
import nbformat

# StructureMap imports
import structuremap
import structuremap.utils
structuremap.utils.set_logger()

from structuremap.processing import (
    download_alphafold_cif,
    download_alphafold_pae,
    format_alphafold_data,
    annotate_accessibility,
    get_smooth_score,
    annotate_proteins_with_idr_pattern,
    get_extended_flexible_pattern,
    get_proximity_pvals,
    perform_enrichment_analysis,
    evaluate_ptm_colocalization,
    extract_motifs_in_proteome
)

from structuremap.plotting import (
    plot_enrichment,
    plot_ptm_colocalization
)


In [4]:
os.chdir("..")

Load protein list

In [5]:
protein_list_path = "data/proteins.txt"

with open(protein_list_path, "r") as f:
    test_proteins = [line.strip() for line in f if line.strip()]

print("Proteins loaded:", len(test_proteins))
test_proteins[:10]


Proteins loaded: 6


['O70468', 'Q02566', 'O55143', 'P05202', 'P16858', 'P48962']

Define folders

In [ ]:
# Temporary folders for AlphaFold data
output_dir = tempfile.gettempdir()

cif_dir = os.path.join(output_dir, "cif")
pae_dir = os.path.join(output_dir, "pae")

os.makedirs(cif_dir, exist_ok=True)
os.makedirs(pae_dir, exist_ok=True)

# Folder for plots in shared drive (SC)
plots_base = r"S:\path\to\folder\in\StructureMap\plots" #place the folder path
structuremap_dir = os.path.join(plots_base, "StructureMap")
plots_dir = os.path.join(structuremap_dir, "plots")

os.makedirs(plots_dir, exist_ok=True)


Download AlphaFold data

In [7]:
valid_cif, invalid_cif, existing_cif = download_alphafold_cif(
    proteins=test_proteins,
    out_folder=cif_dir
)

valid_pae, invalid_pae, existing_pae = download_alphafold_pae(
    proteins=test_proteins,
    out_folder=pae_dir
)


100%|██████████| 6/6 [00:00<00:00, 1648.38it/s]

2026-03-17 14:13:14> Valid proteins: 0
2026-03-17 14:13:14> Invalid proteins: 0
2026-03-17 14:13:14> Existing proteins: 6



100%|██████████| 6/6 [00:00<00:00, 5977.63it/s]

2026-03-17 14:13:14> Valid proteins: 0
2026-03-17 14:13:14> Invalid proteins: 0
2026-03-17 14:13:14> Existing proteins: 6


Format AlphaFold data

In [8]:
alphafold_annotation = format_alphafold_data(
    directory=cif_dir,
    protein_ids=test_proteins
)

100%|██████████| 6/6 [00:03<00:00,  1.54it/s]


Annotate accessibility (pPSE)

In [9]:
# Full sphere
full_acc = annotate_accessibility(
    df=alphafold_annotation,
    max_dist=24,
    max_angle=180,
    error_dir=pae_dir
)

# Partial sphere
partial_acc = annotate_accessibility(
    df=alphafold_annotation,
    max_dist=12,
    max_angle=70,
    error_dir=pae_dir
)

# Merge
alphafold_accessibility = (
    alphafold_annotation
    .merge(full_acc, on=["protein_id","AA","position"], how="left")
    .merge(partial_acc, on=["protein_id","AA","position"], how="left")
)

# Binary accessibility
alphafold_accessibility["high_acc_5"] = (alphafold_accessibility["nAA_12_70_pae"] <= 5).astype(int)
alphafold_accessibility["low_acc_5"]  = (alphafold_accessibility["nAA_12_70_pae"] > 5).astype(int)


100%|██████████| 6/6 [00:00<00:00, 31.77it/s]


Annotate IDRs

In [10]:
alphafold_smooth = get_smooth_score(
    alphafold_accessibility,
    np.array(["nAA_24_180_pae"]),
    [10]
)

alphafold_smooth["IDR"] = (alphafold_smooth["nAA_24_180_pae_smooth10"] <= 34.27).astype(int)

# Short IDRs + extended flexible regions
alphafold_pattern = annotate_proteins_with_idr_pattern(
    alphafold_smooth,
    min_structured_length=80,
    max_unstructured_length=20
)

alphafold_pattern_ext = get_extended_flexible_pattern(
    alphafold_pattern,
    ["flexible_pattern"],
    [5]
)


  0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:01<00:00,  4.73it/s]


Load PTM_table

In [11]:
ptm_file = pd.read_csv("data/processed/quant_pgm_NM_LIMMA_ptm.tsv", sep="\t")

Merge

In [12]:
# Convertmultiplepositions
ptm_file["position"] = ptm_file["position"].astype(str)
ptm_file = ptm_file.assign(position=ptm_file["position"].str.split(";"))
ptm_file = ptm_file.explode("position")
ptm_file["position"] = ptm_file["position"].str.strip().astype(int)

# Merge
alphafold_ptms = alphafold_pattern_ext.merge(
    ptm_file,
    how="left",
    on=["protein_id","AA","position"]
).fillna(0)


In [13]:
ptm_file.columns

Index(['protein_id', 'AA', 'position', 'ox', 'carb', 'me', 'kin', 'diox',
       'rev_ox', 'p', 'triox'],
      dtype='object')

Load PTM dictionary

In [14]:
ptm_site_dict = {
    'p': ['S','T','Y'],
    'me': ['M','E','H','T','C','K','N','Q','R','I','L','D','S'],
    'ox': ['M','Q','W','H','Y','C','D','K','N','P','R'],
    'diox': ['W','F','C','M','P','R','K','Y'],
    'triox': ['C','W','Y','F','M'],
    'rev_ox': ['C'],
    'carb': ['C'],
    'kin': ['W']
}


PTM colocalization

In [16]:
#Ox vs. all
p_colocalization_ox = evaluate_ptm_colocalization(
    df=alphafold_ptms, 
    ptm_target='ox',
    ptm_types=['p','me','diox','triox','rev_ox','carb','kin'], 
    ptm_dict=ptm_site_dict,
    pae_dir=pae_dir,
    min_dist=1,
    max_dist=35,
    dist_step=5
)

fig_ox = plot_ptm_colocalization(
    p_colocalization_ox,
    context="3D",
    plot_width=1500
)



  0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 22.48it/s]


In [17]:
#Rev_ox vs all
p_colocalization_oxrev = evaluate_ptm_colocalization(
    df=alphafold_ptms, 
    ptm_target='rev_ox',
    ptm_types=['p','me','diox','triox','ox','carb','kin'], 
    ptm_dict=ptm_site_dict,
    pae_dir=pae_dir,
    min_dist=1,
    max_dist=35,
    dist_step=5
)

fig_oxrev = plot_ptm_colocalization(
    p_colocalization_oxrev,
    context="3D",
    plot_width=1500
)




100%|██████████| 6/6 [00:00<00:00, 44.06it/s]


Enrichment analysis

In [19]:
enrichment = perform_enrichment_analysis(
    df=alphafold_ptms,
    ptm_types=['p','me','ox','diox','triox','rev_ox','carb','kin'],
    rois=['IDR'],
    ptm_site_dict=ptm_site_dict,
    quality_cutoffs=[0]
)

fig_enrich = plot_enrichment(
    data=enrichment,
    ptm_select=['p','me','ox','diox','triox','rev_ox','carb','kin'],
    roi_select=['IDR']
)



c:\Users\irodrigueza\project_ptms\env_py38\lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Export final table (alphafold_ptms)

In [23]:
alphafold_ptms.to_csv(
    "data/processed/alphafold_ptms.tsv",
    sep="\t",
    index=False
)

print("Final table saved in: data/processed/alphafold_ptms.tsv")

alphafold_ptms.to_csv(
    os.path.join(plots_dir, "alphafold_ptms.tsv"),
    sep="\t",
    index=False
)

print("Final table also saved in:", os.path.join(plots_dir, "alphafold_ptms.tsv"))


Final table saved in: data/processed/alphafold_ptms.tsv
Final table also saved in: S:\U_Proteomica\LABS\LAB_BI\Proyecto_IR_PTMs\StructureMap_output\plots\alphafold_ptms.tsv
